# Entity Classification Pipeline — Progressive Candidate Narrowing

Assigns every entity in `doc_db.json` a single **system** and **subsystem** label
using a progressive approach: build broad candidate sets first, then narrow them
using structural and embedding signals.

### Input Resources
| File | Purpose |
|------|---------|
| `doc_db.backup.json` | Flat entity database (loaded fresh for idempotency) |
| `code_graph.json` | Raw entity records with file-location metadata |
| `code_graph.gml` | GML dependency graph (calls/uses/inherits/includes) |
| `taxonomy.json` | Hand-curated system → subsystem hierarchy (target label space) |
| `file_to_systems.json` | Hand-curated source-file → system(s) mapping |

### Pipeline

| Phase | Produces | Strategy |
|-------|----------|----------|
| **0. Embeddings** | SBERT vectors per entity | Cached to disk; used in narrowing and final decision |
| **1. File-path seed** | `sys_cands` per entity | Seed from `file_to_systems.json`; propagate to members |
| **2. Leiden gap-fill + utility** | Gap-filled candidates, utility set | Unseeded nodes get majority-voted system from cluster peers; seeded nodes unchanged |
| **3. Promote system candidates** | Compound candidates | Members → compounds at ≥30% threshold |
| **4. Narrow system candidates** | Reduced candidate sets | k-NN embedding vote drops weak candidates |
| **5. Subsystem clustering** | `sub_cands` per entity | Per-system Leiden on candidate members; auto-naming |
| **6. Promote + narrow subsystems** | Reduced (sys, sub) pairs | Same promote/narrow pattern as systems |
| **7. Final decision + save** | `doc.system`, `doc.subsystem` | k-NN picks best pair; else General; save + validate |

### Key Design Principles
- **Defer hard decisions**: accumulate candidate sets, narrow progressively
- **Members are the unit of analysis**: they carry structural edges (`calls/uses/inherits`)
- **Compounds inherit from members**: majority vote promotion at 30% threshold
- **Utility nodes identified structurally**: via fan-in/fan-out scoring across clusters
- **Embeddings for narrowing and final call**: k-NN in full embedding space (not centroids)

In [1]:
%load_ext autoreload
%autoreload 2

In [15]:
# ── Setup: load all resources ────────────────────────────────────────
from __future__ import annotations

import json
import sys
import pickle
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import networkx as nx
from sklearn.metrics.pairwise import cosine_similarity as cos_sim

HERE = Path(".").resolve()
CLUSTERING_DIR = HERE / "clustering"
if str(CLUSTERING_DIR) not in sys.path:
    sys.path.insert(0, str(CLUSTERING_DIR))

import clustering.doc_db as ddb
import clustering.doxygen_graph as dg
import clustering.doxygen_parse as dp
import clustering.subsystem_utils as su
import clustering.structural_clustering as sc
import clustering.subsystem_identifier as si

# ── Load doc_db from BACKUP (so re-runs are idempotent) ─────────────
db = ddb.DocumentDB()
db.load(ddb.DOC_DB_BACKUP_PATH)
print(f"DocumentDB loaded: {db.count()} entities")

# ── Build CID → source file(s) lookup from code_graph.json ─────────
with open(CLUSTERING_DIR / "code_graph.json") as f:
    _cg_raw = json.load(f)
CID_TO_FILES: dict[str, set[str]] = defaultdict(set)
for _e in _cg_raw:
    _cid = _e["id"].get("compound", "")
    for _loc_key in ("body", "file"):
        _loc = _e.get(_loc_key)
        if _loc and _loc.get("fn"):
            CID_TO_FILES[_cid].add(_loc["fn"])
print(f"code_graph.json: {len(_cg_raw)} entities, {len(CID_TO_FILES)} with file info")

# ── Load the GML dependency graph ───────────────────────────────────
graph = dg.load_graph(CLUSTERING_DIR / "code_graph.gml")
print(f"GML graph: {len(graph.nodes)} nodes, {len(graph.edges)} edges")

# ── Load entity_db ──────────────────────────────────────────────────
entity_db = su.load_entity_db(su.DEFAULT_DB_PATH)
print(f"Entity database: {len(entity_db.entities)} entities")

# ── Strip any prior classification (clean slate) ────────────────────
for doc in db.all_docs():
    doc.system = None
    doc.subsystem = None
    doc.system_candidates = None
    doc.subsystem_candidates = None

# ── Build reverse indexes ───────────────────────────────────────────
# member_hash → doc for quick lookup
hash_to_doc: dict[str, ddb.Document] = {}
# compound → set of member hashes in doc_db
compound_to_members: dict[str, set[str]] = defaultdict(set)
# compounds that have their own compound-only doc_db entry (classes, structs, etc.)
compound_has_doc: set[str] = set()

for cid, sig, doc in db.items():
    _, member_hash = dp.EntityID.split(doc.mid)
    if member_hash:
        hash_to_doc[member_hash] = doc
        compound_to_members[doc.cid].add(member_hash)
    else:
        compound_has_doc.add(doc.cid)

member_hashes_in_graph = {h for h in hash_to_doc if h in graph}

print(f"\nReverse indexes built:")
print(f"  member docs:         {len(hash_to_doc)} ({len(member_hashes_in_graph)} in graph)")
print(f"  compound-only docs:  {len(compound_has_doc)}")
print(f"  compounds w/ members:{len(compound_to_members)}")
print("All prior classifications cleared.")

DocumentDB loaded: 5293 entities
code_graph.json: 5305 entities, 447 with file info


2026-03-13 09:36:12.700 | INFO     | clustering.doxygen_graph:load_graph:421 - Graph loaded from /Users/QTE2333/repos/legacy/.ai/gen_docs/clustering/code_graph.gml, nodes: 14507, edges: 44544


GML graph: 14507 nodes, 44544 edges
Entity database: 5305 entities

Reverse indexes built:
  member docs:         5015 (4240 in graph)
  compound-only docs:  216
  compounds w/ members:329
All prior classifications cleared.


In [16]:
# ── Load taxonomy and file-to-system mappings from JSON ──────────────
with open(HERE / "taxonomy.json") as f:
    TAXONOMY: dict[str, list[str]] = json.load(f)

with open(HERE / "file_to_systems.json") as f:
    FILE_TO_SYSTEMS: dict[str, list[str]] = json.load(f)

# ── Derive convenience structures ───────────────────────────────────
ALL_SYSTEMS = list(TAXONOMY.keys())
SYSTEM_SET = set(ALL_SYSTEMS)
SUBSYSTEM_TO_SYSTEM = {sub: sys for sys, subs in TAXONOMY.items() for sub in subs}

print(f"Taxonomy: {len(ALL_SYSTEMS)} systems, {len(SUBSYSTEM_TO_SYSTEM)} subsystems")
print(f"File mappings: {len(FILE_TO_SYSTEMS)} files")

Taxonomy: 23 systems, 150 subsystems
File mappings: 214 files


## Phase 0: Compute & Cache SBERT Embeddings

Generate a dense embedding vector for every entity in `doc_db` from its Doxygen
documentation text using Sentence-BERT (`all-MiniLM-L6-v2`).

Embeddings are cached to `embeddings_cache.pkl` so subsequent runs skip
the encoding step entirely. These vectors are used later for k-NN narrowing
(Phases 4, 6) and the final assignment decision (Phase 7).

In [17]:
# ── Phase 0: Compute & cache embeddings ──────────────────────────────
import openai_embeddings as oai_emb

EMBEDDING_CACHE = CLUSTERING_DIR / "embeddings_cache.pkl"
_skip_encode = False

if EMBEDDING_CACHE.exists():
    with open(EMBEDDING_CACHE, "rb") as f:
        all_embeddings: dict[str, np.ndarray] = pickle.load(f)
    sample = next(iter(all_embeddings.values()))
    print(f"Loaded cached embeddings: {len(all_embeddings)} entities ({sample.shape[0]}d)")
    _skip_encode = True

if not _skip_encode:
    texts: list[str] = []
    keys: list[str] = []
    for cid, sig, doc in db.items():
        text = doc.to_doxygen()
        if text:
            texts.append(text)
            keys.append(doc.mid)

    print(f"Encoding {len(texts)} entities with {oai_emb.MODEL}...")
    vectors = oai_emb.embed_texts(texts, show_progress=True)
    all_embeddings = {k: v for k, v in zip(keys, vectors)}

    with open(EMBEDDING_CACHE, "wb") as f:
        pickle.dump(all_embeddings, f)
    print(f"Cached {len(all_embeddings)} embeddings ({vectors.shape[1]}d) to {EMBEDDING_CACHE}")

# ── Build embedding lookup by member hash ────────────────────────────
# all_embeddings is keyed by doc.mid (= "compound_hash" for members)
# We also need lookup by member_hash and by compound_id for compound-only docs
emb_by_key: dict[str, np.ndarray] = {}  # member_hash or compound_id → embedding

for cid, sig, doc in db.items():
    emb = all_embeddings.get(doc.mid)
    if emb is None:
        continue
    _, mhash = dp.EntityID.split(doc.mid)
    key = mhash if mhash else doc.cid
    emb_by_key[key] = emb

print(f"Embedding lookup: {len(emb_by_key)} entities by member_hash/compound_id")

Loaded cached embeddings: 5293 entities
Embedding lookup: 5231 entities by member_hash/compound_id


## Phase 1: File-Path Seed

Assign initial `system_candidates` to every entity from the hand-curated
`file_to_systems.json` mapping.  Since entities share a compound ID with their
source file, both compound-only docs (classes, structs) and member docs
(functions, variables) inherit candidates from the same compound.

In [18]:
# ── Phase 1: File-path seed ──────────────────────────────────────────
# sys_cands: the central working dict — maps member_hash or compound_id to
# a set of candidate system names.  We use a plain dict (not doc attributes)
# so we can manipulate freely before the final write.

sys_cands: dict[str, set[str]] = defaultdict(set)

seeded = 0
for cid, sig, doc in db.items():
    files = CID_TO_FILES.get(doc.cid, set())
    if not files:
        continue
    candidates: set[str] = set()
    for fn in files:
        for s in FILE_TO_SYSTEMS.get(fn, []):
            candidates.add(s)
    if candidates:
        _, mhash = dp.EntityID.split(doc.mid)
        key = mhash if mhash else doc.cid
        sys_cands[key] |= candidates
        seeded += 1

# Summary
keys_with_cands = len(sys_cands)
multi = sum(1 for v in sys_cands.values() if len(v) > 1)
print(f"File-path seed: {seeded}/{db.count()} entities seeded")
print(f"Unique keys with candidates: {keys_with_cands} (multi-system: {multi})")

File-path seed: 5293/5293 entities seeded
Unique keys with candidates: 5231 (multi-system: 1137)


## Phase 2: Leiden Gap-Fill + Utility Identification

Two operations on the full dependency graph:

1. **Leiden community detection** → group structurally connected nodes into clusters.
   **Seeded nodes keep their file-path candidates unchanged.**
   **Unseeded nodes** (no file-path match) receive the majority-voted system(s) from
   their seeded cluster peers. This fills gaps without destroying the high-confidence
   file-path signal.

2. **Utility node identification** using `identify_utility_nodes` which scores each node on
   fan-in/fan-out across clusters. Nodes with high cross-cluster fan-in and low fan-out
   are flagged as utilities and excluded from further broadening. They will get
   `system="Utilities"` at final assignment.

In [19]:
# ── Phase 2: Leiden gap-fill + utility identification ────────────────

# Remember which keys were seeded in Phase 1 (file-path signal is high-confidence)
seeded_keys: set[str] = set(sys_cands.keys())

# 2a. Leiden clustering on the full graph
leiden_clustering = sc.apply_leiden_clustering(graph, resolution=1.0)
n_clusters = len(set(leiden_clustering.values()))
print(f"Leiden: {n_clusters} clusters over {len(leiden_clustering)} nodes")

# 2b. Gap-fill: only UNSEEDED nodes receive candidates from cluster peers
# Seeded nodes keep their file-path candidates untouched.
# Unseeded nodes get majority-voted system(s) from seeded peers in same cluster.
cluster_members: dict[int, set[str]] = defaultdict(set)
for node, cl_id in leiden_clustering.items():
    cluster_members[cl_id].add(node)

gap_filled = 0
CLUSTER_VOTE_FRAC = 0.20  # system must have ≥20% support among seeded peers

for cl_id, nodes in cluster_members.items():
    # Count system votes from SEEDED peers only
    peer_votes: Counter = Counter()
    seeded_in_cluster = 0
    for n in nodes:
        if n in seeded_keys:
            seeded_in_cluster += 1
            for s in sys_cands.get(n, set()):
                peer_votes[s] += 1

    if not peer_votes:
        continue

    # Determine which systems have meaningful support
    min_vote = max(1, int(seeded_in_cluster * CLUSTER_VOTE_FRAC))
    strong_systems = {s for s, cnt in peer_votes.items() if cnt >= min_vote}

    if not strong_systems:
        # Fallback: just use the most common system
        strong_systems = {peer_votes.most_common(1)[0][0]}

    # Assign to UNSEEDED nodes only
    for n in nodes:
        if n not in seeded_keys and n not in sys_cands:
            sys_cands[n] = set(strong_systems)
            gap_filled += 1

docdb_with_cands = sum(1 for h in hash_to_doc if sys_cands.get(h))
print(f"Gap-filled: {gap_filled} unseeded nodes received candidates from cluster peers")
print(f"Seeded nodes unchanged: {len(seeded_keys)}")
print(f"Doc_db members with system candidates: {docdb_with_cands}/{len(hash_to_doc)}")

# 2c. Utility identification via fan-in/fan-out scoring
etypes = ('calls', 'uses')

def calc_utility_score(score: dict[str, float]) -> bool:
    return (
        ((sum(score[f'fan_in_{etype}'] for etype in etypes) +
          sum(score[f'fan_in_clusters_{etype}'] for etype in etypes)) / 2)
        * (1 - (sum(score[f'fan_out_{etype}'] for etype in etypes) +
                 sum(score[f'fan_out_clusters_{etype}'] for etype in etypes) +
                 sum(score[f'overlap_{etype}'] for etype in etypes)) / 2)
    )

UTILITY_THRESHOLD = 0.226
utility_nodes: set[str] = sc.identify_utility_nodes(
    graph,
    utility_threshold_fn=lambda score: calc_utility_score(score) > UTILITY_THRESHOLD,
    initial_clustering=leiden_clustering,
)
utility_in_docdb = utility_nodes & set(hash_to_doc.keys())
print(f"\nUtility nodes: {len(utility_nodes)} total, {len(utility_in_docdb)} in doc_db")

# Remove utility nodes from sys_cands — they'll be system="Utilities" at final decision
for node in utility_nodes:
    sys_cands.pop(node, None)

# Summary after gap-fill + utility removal
single = sum(1 for v in sys_cands.values() if len(v) == 1)
multi = sum(1 for v in sys_cands.values() if len(v) > 1)
print(f"\nAfter gap-fill + utility removal:")
print(f"  single-system:  {single}")
print(f"  multi-system:   {multi}")
print(f"  total with cands: {single + multi}")

2026-03-13 09:36:28.791 | INFO     | clustering.structural_clustering:apply_leiden_clustering:134 - Applying Leiden clustering with resolution=1.0 on graph with 14507 nodes and 44544 edges
2026-03-13 09:36:28.814 | INFO     | clustering.structural_clustering:apply_leiden_clustering:152 - Filtered graph created with 4154 nodes and 26596 edges
2026-03-13 09:36:28.815 | INFO     | clustering.structural_clustering:apply_leiden_clustering:155 - Converting NetworkX graph to iGraph format
2026-03-13 09:36:28.819 | INFO     | clustering.structural_clustering:apply_leiden_clustering:163 - Running Leiden community detection algorithm
2026-03-13 09:36:29.143 | INFO     | clustering.structural_clustering:apply_leiden_clustering:181 - Leiden clustering completed in 0.36s. Found 26 communities
2026-03-13 09:36:29.146 | INFO     | clustering.structural_clustering:identify_utility_nodes:35 - Identifying utility nodes
2026-03-13 09:36:29.147 | INFO     | clustering.structural_clustering:identify_utilit

Leiden: 26 clusters over 4154 nodes
Gap-filled: 82 unseeded nodes received candidates from cluster peers
Seeded nodes unchanged: 5231
Doc_db members with system candidates: 5015/5015


2026-03-13 09:38:39.976 | INFO     | clustering.structural_clustering:identify_utility_nodes:103 - Utility scores saved to utility_scores.json
2026-03-13 09:38:39.979 | INFO     | clustering.structural_clustering:identify_utility_nodes:119 - Identified 59 utility nodes in 130.83s



Utility nodes: 59 total, 51 in doc_db

After gap-fill + utility removal:
  single-system:  4100
  multi-system:   1154
  total with cands: 5254


## Phase 3: Promote System Candidates to Compounds

Compounds with individual doc_db entries (classes, structs, enums) don't have
structural edges themselves — their members do. This phase promotes system
candidates **upward** from members to their parent compound.

A system becomes a compound candidate if ≥30% of its non-utility members carry it.
This is intentionally permissive (broadening); narrowing happens in the next phase.

In [20]:
# ── Phase 3: Promote system candidates from members to compounds ─────
PROMOTE_THRESHOLD = 0.30

promoted = 0
for compound_id in compound_has_doc:
    members = compound_to_members.get(compound_id, set())
    if not members:
        continue

    # Count how many non-utility members carry each system candidate
    sys_votes: Counter = Counter()
    eligible_count = 0
    for mhash in members:
        if mhash in utility_nodes:
            continue
        mc = sys_cands.get(mhash, set())
        if mc:
            eligible_count += 1
            for s in mc:
                sys_votes[s] += 1

    if eligible_count == 0:
        continue

    # Promote systems above threshold
    compound_cands: set[str] = set()
    for sys_name, count in sys_votes.items():
        if count / eligible_count >= PROMOTE_THRESHOLD:
            compound_cands.add(sys_name)

    if compound_cands:
        old = len(sys_cands.get(compound_id, set()))
        sys_cands[compound_id] |= compound_cands
        if len(sys_cands[compound_id]) > old:
            promoted += 1

print(f"Promoted system candidates to {promoted}/{len(compound_has_doc)} compound docs")

# Summary
total_keys = len(sys_cands)
single = sum(1 for v in sys_cands.values() if len(v) == 1)
multi = sum(1 for v in sys_cands.values() if len(v) > 1)
no_cand_members = sum(1 for h in hash_to_doc if not sys_cands.get(h) and h not in utility_nodes)
no_cand_compounds = sum(1 for c in compound_has_doc if not sys_cands.get(c))
print(f"After promotion: single={single}, multi={multi}")
print(f"Still without candidates: {no_cand_members} members, {no_cand_compounds} compounds")

Promoted system candidates to 6/216 compound docs
After promotion: single=4106, multi=1154
Still without candidates: 0 members, 0 compounds


## Phase 4: Narrow System Candidates (k-NN)

Reduce multi-system candidate sets using k-nearest-neighbor voting in SBERT
embedding space.

For each entity with >1 system candidate, find its k nearest neighbors
(among all entities with candidates).  Only keep candidates that appear
in at least 20% of those neighbors.  If no candidate clears the threshold,
keep only the top-voted one.

Two entity classes participate in narrowing:
- **Compounds** with individual doc_db entries (classes, structs)
- **Members** whose compounds do NOT have doc_db entries (file-scope functions, globals)

In [21]:
# ── Phase 4: Narrow system candidates with k-NN ─────────────────────
K_NEIGHBORS = 15
KNN_SUPPORT_FRAC = 0.20  # candidate must appear in ≥20% of neighbors

# Determine which entities participate in narrowing:
# - Compounds with doc_db entries
# - Members whose compound does NOT have a doc_db entry (file-scope entities)
narrowing_keys: set[str] = set()
# Compounds with doc entries
narrowing_keys |= {c for c in compound_has_doc if sys_cands.get(c)}
# File-scope members (compound not in compound_has_doc)
for mhash, doc in hash_to_doc.items():
    if doc.cid not in compound_has_doc and sys_cands.get(mhash):
        narrowing_keys.add(mhash)
print(f"Entities eligible for narrowing: {len(narrowing_keys)}")

# Build k-NN pool: all entities with embedding + system candidates
knn_keys: list[str] = []
knn_vecs: list[np.ndarray] = []
knn_cands: list[set[str]] = []

for key, cands in sys_cands.items():
    if not cands:
        continue
    emb = emb_by_key.get(key)
    if emb is not None:
        knn_keys.append(key)
        knn_vecs.append(emb)
        knn_cands.append(cands)

knn_matrix = np.stack(knn_vecs)
print(f"k-NN pool: {len(knn_keys)} entities with embeddings + candidates")

# Build index for fast lookup
knn_key_to_idx = {k: i for i, k in enumerate(knn_keys)}

# Compute full cosine similarity matrix (entities are already L2-normalized)
print("Computing similarity matrix...")
sim_matrix = cos_sim(knn_matrix)

# Narrow multi-candidate entities via k-NN vote
narrowed = 0
min_support = max(1, int(K_NEIGHBORS * KNN_SUPPORT_FRAC))

for key in narrowing_keys:
    cands = sys_cands.get(key, set())
    if len(cands) <= 1:
        continue
    idx = knn_key_to_idx.get(key)
    if idx is None:
        continue

    # Get k nearest neighbors (exclude self)
    sims = sim_matrix[idx]
    neighbor_indices = np.argsort(-sims)[1:K_NEIGHBORS+1]

    # Count neighbor support for each of OUR candidates
    support: Counter = Counter()
    for j in neighbor_indices:
        for s in knn_cands[j]:
            if s in cands:
                support[s] += 1

    if support:
        strong = {s for s, c in support.items() if c >= min_support}
        if strong and len(strong) < len(cands):
            sys_cands[key] = strong
            narrowed += 1
        elif not strong:
            sys_cands[key] = {support.most_common(1)[0][0]}
            narrowed += 1

print(f"Narrowed system candidates for {narrowed} entities")
single = sum(1 for v in sys_cands.values() if len(v) == 1)
multi = sum(1 for v in sys_cands.values() if len(v) > 1)
print(f"After narrowing: single={single}, multi={multi}")

Entities eligible for narrowing: 2649
k-NN pool: 5180 entities with embeddings + candidates
Computing similarity matrix...
Narrowed system candidates for 178 entities
After narrowing: single=4228, multi=1032


## Phase 5: Per-System Subsystem Clustering

For each system, gather all **member** graph nodes that have it as a candidate,
then run Leiden subclustering on that per-system subgraph.

Members with multiple system candidates will participate in subclustering for
each of those systems, yielding subsystem candidates in multiple systems.
This is intentional — the final decision (Phase 7) will pick the best combination.

Unclustered members (no `calls/uses/inherits` edges within their system subgraph)
inherit a subsystem from siblings in the same compound via majority vote.

In [22]:
# ── Phase 5: Per-system subsystem clustering ─────────────────────────

# Build system → candidate member hashes (only members, not compounds)
system_to_candidate_members: dict[str, set[str]] = defaultdict(set)

for key, cands in sys_cands.items():
    if key in hash_to_doc and key in graph:  # must be a member in the graph
        for s in cands:
            system_to_candidate_members[s].add(key)

print("Per-system candidate members (in graph):")
for sys_name, members in sorted(system_to_candidate_members.items(), key=lambda x: -len(x[1])):
    print(f"  {sys_name:30s} {len(members):5d}")

# sub_cands: key → set of (system, subsystem_name) candidate pairs
sub_cands: dict[str, set[tuple[str, str]]] = defaultdict(set)
# Track which nodes belong to each subsystem (for centroids/k-NN later)
subsystem_nodes: dict[tuple[str, str], set[str]] = defaultdict(set)

for sys_name, sys_members in system_to_candidate_members.items():
    live_nodes = [m for m in sys_members if m in graph]

    if len(live_nodes) < 3:
        sub_name = f"{sys_name} Core"
        for n in live_nodes:
            sub_cands[n].add((sys_name, sub_name))
            subsystem_nodes[(sys_name, sub_name)].add(n)
        continue

    subgraph = graph.subgraph(live_nodes).copy()
    sub_clustering = sc.apply_leiden_clustering(subgraph, resolution=1.0)

    # Group by subcluster
    cluster_nodes_map: dict[int, set[str]] = defaultdict(set)
    for node, c in sub_clustering.items():
        cluster_nodes_map[c].add(node)

    unclustered = set(live_nodes) - set(sub_clustering.keys())

    # Name each subcluster
    for c_id, c_nodes in cluster_nodes_map.items():
        if not c_nodes:
            continue

        entity_names: list[str] = []
        entity_types: Counter = Counter()
        for nid in c_nodes:
            entity = su.get_entity(entity_db, nid)
            if entity:
                entity_names.append(entity.name)
                entity_types[entity.kind] += 1

        term_counter = si.extract_significant_terms(entity_names)
        classification = si.classify_subsystem(term_counter)
        sub_name = si.determine_subsystem_name(term_counter, entity_types, classification)

        for nid in c_nodes:
            sub_cands[nid].add((sys_name, sub_name))
            subsystem_nodes[(sys_name, sub_name)].add(nid)

    # Unclustered: sibling vote from same compound
    for nid in unclustered:
        parent_compounds = []
        for succ in graph.successors(nid):
            edge_data = graph.get_edge_data(nid, succ)
            if edge_data and 'contained_by' in edge_data:
                parent_compounds.append(succ)

        sibling_subs: Counter = Counter()
        for parent in parent_compounds:
            for sibling in graph.predecessors(parent):
                if sibling == nid:
                    continue
                edge_data = graph.get_edge_data(sibling, parent)
                if edge_data and 'contained_by' in edge_data:
                    for pair in sub_cands.get(sibling, set()):
                        if pair[0] == sys_name:
                            sibling_subs[pair] += 1

        if sibling_subs:
            best = sibling_subs.most_common(1)[0][0]
            sub_cands[nid].add(best)
            subsystem_nodes[best].add(nid)
        else:
            gen = (sys_name, f"{sys_name} General")
            sub_cands[nid].add(gen)
            subsystem_nodes[gen].add(nid)

total_pairs = sum(len(v) for v in sub_cands.values())
unique_subs = len(subsystem_nodes)
single_pair = sum(1 for v in sub_cands.values() if len(v) == 1)
multi_pair = sum(1 for v in sub_cands.values() if len(v) > 1)
print(f"\nSubsystem candidates: {len(sub_cands)} nodes, {total_pairs} total (sys,sub) pairs")
print(f"Unique subsystems: {unique_subs}")
print(f"Nodes with single (sys,sub): {single_pair}, multiple: {multi_pair}")

2026-03-13 09:38:54.808 | INFO     | clustering.structural_clustering:apply_leiden_clustering:134 - Applying Leiden clustering with resolution=1.0 on graph with 1222 nodes and 236 edges
2026-03-13 09:38:54.808 | INFO     | clustering.structural_clustering:apply_leiden_clustering:152 - Filtered graph created with 230 nodes and 236 edges
2026-03-13 09:38:54.809 | INFO     | clustering.structural_clustering:apply_leiden_clustering:155 - Converting NetworkX graph to iGraph format
2026-03-13 09:38:54.809 | INFO     | clustering.structural_clustering:apply_leiden_clustering:163 - Running Leiden community detection algorithm
2026-03-13 09:38:54.811 | INFO     | clustering.structural_clustering:apply_leiden_clustering:181 - Leiden clustering completed in 0.00s. Found 16 communities


Per-system candidate members (in graph):
  Data Tables                     1222
  Object System                    626
  Command Interpreter              604
  World System                     545
  Character Data Model             456
  Networking                       303
  Persistence                      296
  Utilities                        293
  Combat System                    275
  Magic System                     243
  Social & Communication           229
  Admin & Builder Commands         171
  MobProg System                   167
  Organizations & PvP              152
  Affect System                    132
  Quests                           116
  Notes & Editor                   106
  Character Progression             94
  Memory & Garbage Collection       93
  Game Engine                       78
  Help System                       53
  Economy                           39
  Event Dispatcher                  15


2026-03-13 09:38:55.035 | INFO     | clustering.structural_clustering:apply_leiden_clustering:134 - Applying Leiden clustering with resolution=1.0 on graph with 545 nodes and 784 edges
2026-03-13 09:38:55.036 | INFO     | clustering.structural_clustering:apply_leiden_clustering:152 - Filtered graph created with 390 nodes and 781 edges
2026-03-13 09:38:55.037 | INFO     | clustering.structural_clustering:apply_leiden_clustering:155 - Converting NetworkX graph to iGraph format
2026-03-13 09:38:55.037 | INFO     | clustering.structural_clustering:apply_leiden_clustering:163 - Running Leiden community detection algorithm
2026-03-13 09:38:55.046 | INFO     | clustering.structural_clustering:apply_leiden_clustering:181 - Leiden clustering completed in 0.01s. Found 17 communities
2026-03-13 09:38:55.057 | INFO     | clustering.structural_clustering:apply_leiden_clustering:134 - Applying Leiden clustering with resolution=1.0 on graph with 456 nodes and 960 edges
2026-03-13 09:38:55.058 | INFO 


Subsystem candidates: 4189 nodes, 6308 total (sys,sub) pairs
Unique subsystems: 300
Nodes with single (sys,sub): 3258, multiple: 931


## Phase 6: Promote + Narrow Subsystem Candidates

Same two-step pattern as system candidates:

1. **Promote** (sys, sub) pairs from members to their parent compound at ≥30% threshold.
2. **Narrow** multi-candidate entities via k-NN: for each entity with >1 (sys, sub) pair,
   find its k nearest neighbors and keep only well-supported pairs.

In [23]:
# ── Phase 6: Promote + narrow subsystem candidates ───────────────────

# 6a. Promote (sys, sub) pairs from members to compounds
promoted_sub = 0
for compound_id in compound_has_doc:
    members = compound_to_members.get(compound_id, set())
    if not members:
        continue

    pair_votes: Counter = Counter()
    eligible = 0
    for mhash in members:
        if mhash in utility_nodes:
            continue
        mc = sub_cands.get(mhash, set())
        if mc:
            eligible += 1
            for pair in mc:
                pair_votes[pair] += 1

    if eligible == 0:
        continue

    compound_sub: set[tuple[str, str]] = set()
    for pair, count in pair_votes.items():
        if count / eligible >= PROMOTE_THRESHOLD:
            compound_sub.add(pair)

    if compound_sub:
        old = len(sub_cands.get(compound_id, set()))
        sub_cands[compound_id] |= compound_sub
        if len(sub_cands[compound_id]) > old:
            promoted_sub += 1

print(f"Promoted subsystem candidates to {promoted_sub}/{len(compound_has_doc)} compounds")

# 6b. Narrow multi-candidate entities via k-NN
# Build lookup: knn_key → set of (sys, sub) for the k-NN pool
knn_sub_cands: list[set[tuple[str, str]]] = [
    sub_cands.get(k, set()) for k in knn_keys
]

narrowed_sub = 0
for key in narrowing_keys:
    cands = sub_cands.get(key, set())
    if len(cands) <= 1:
        continue
    idx = knn_key_to_idx.get(key)
    if idx is None:
        continue

    sims = sim_matrix[idx]
    neighbor_indices = np.argsort(-sims)[1:K_NEIGHBORS+1]

    # Count neighbor support for each of OUR (sys, sub) candidates
    support: Counter = Counter()
    for j in neighbor_indices:
        for pair in knn_sub_cands[j]:
            if pair in cands:
                support[pair] += 1

    if support:
        strong = {p for p, c in support.items() if c >= min_support}
        if strong and len(strong) < len(cands):
            sub_cands[key] = strong
            narrowed_sub += 1
        elif not strong:
            sub_cands[key] = {support.most_common(1)[0][0]}
            narrowed_sub += 1

print(f"Narrowed subsystem candidates for {narrowed_sub} entities")
single_sub = sum(1 for v in sub_cands.values() if len(v) == 1)
multi_sub = sum(1 for v in sub_cands.values() if len(v) > 1)
print(f"After narrowing: single-pair={single_sub}, multi-pair={multi_sub}")

Promoted subsystem candidates to 216/216 compounds
Narrowed subsystem candidates for 381 entities
After narrowing: single-pair=3719, multi-pair=686


## Phase 7: Final Decision + Save

Each entity now has a set of `(system, subsystem)` candidate pairs (possibly empty).
This phase makes the final assignment:

1. **Utility nodes** → `system="Utilities"`, `subsystem="Utilities"`
2. **Single candidate** → assign directly
3. **Multiple candidates** → k-NN vote in embedding space picks the best pair
4. **System candidates only** (no subsystem) → assign system, `subsystem="[System] General"`
5. **No candidates** → `system="General"`, `subsystem="General"`

After all assignments, persist `doc_db.json` and print validation statistics.

In [24]:
# ── Phase 7: Final decision ──────────────────────────────────────────

# For entities with multiple candidates, resolve with k-NN vote
# (same mechanism as narrowing, but forced to pick exactly one)

# Resolve multi-candidate entities via k-NN
resolved_by_knn = 0
for key in list(sub_cands.keys()):
    cands = sub_cands[key]
    if len(cands) <= 1:
        continue
    idx = knn_key_to_idx.get(key)
    if idx is None:
        # No embedding → pick first
        sub_cands[key] = {next(iter(cands))}
        continue

    sims = sim_matrix[idx]
    neighbor_indices = np.argsort(-sims)[1:K_NEIGHBORS+1]

    support: Counter = Counter()
    for j in neighbor_indices:
        nkey = knn_keys[j]
        for pair in sub_cands.get(nkey, set()):
            if pair in cands:
                support[pair] += 1

    if support:
        sub_cands[key] = {support.most_common(1)[0][0]}
        resolved_by_knn += 1
    else:
        sub_cands[key] = {next(iter(cands))}

print(f"Resolved {resolved_by_knn} multi-candidate entities via k-NN")

# Also resolve multi-system entities that had no subsystem candidates
for key in list(sys_cands.keys()):
    cands = sys_cands[key]
    if len(cands) <= 1:
        continue
    idx = knn_key_to_idx.get(key)
    if idx is None:
        sys_cands[key] = {next(iter(cands))}
        continue
    sims = sim_matrix[idx]
    neighbor_indices = np.argsort(-sims)[1:K_NEIGHBORS+1]
    support: Counter = Counter()
    for j in neighbor_indices:
        for s in knn_cands[j]:
            if s in cands:
                support[s] += 1
    if support:
        sys_cands[key] = {support.most_common(1)[0][0]}
    else:
        sys_cands[key] = {next(iter(cands))}

# ── Assign to doc objects ────────────────────────────────────────────
utility_count = 0
assigned_from_sub = 0
assigned_sys_only = 0
general_count = 0

for cid, sig, doc in db.items():
    _, mhash = dp.EntityID.split(doc.mid)
    key = mhash if mhash else doc.cid

    # 1. Utility nodes
    if key in utility_nodes:
        doc.system = "Utilities"
        doc.subsystem = "Utilities"
        utility_count += 1
        continue

    # 2. Has (system, subsystem) candidates
    sc_set = sub_cands.get(key, set())
    if sc_set:
        best_sys, best_sub = next(iter(sc_set))
        doc.system = best_sys
        doc.subsystem = best_sub
        assigned_from_sub += 1
        continue

    # 3. Has system candidates but no subsystem
    sy_set = sys_cands.get(key, set())
    if sy_set:
        best_sys = next(iter(sy_set))
        doc.system = best_sys
        doc.subsystem = f"{best_sys} General"
        assigned_sys_only += 1
        continue

    # 4. Nothing — try to inherit from compound
    if mhash and doc.cid in compound_has_doc:
        parent_sub = sub_cands.get(doc.cid, set())
        if parent_sub:
            ps, pn = next(iter(parent_sub))
            doc.system = ps
            doc.subsystem = pn
            assigned_from_sub += 1
            continue
        parent_sys = sys_cands.get(doc.cid, set())
        if parent_sys:
            ps = next(iter(parent_sys))
            doc.system = ps
            doc.subsystem = f"{ps} General"
            assigned_sys_only += 1
            continue

    # 5. Fallback
    doc.system = "General"
    doc.subsystem = "General"
    general_count += 1

total = utility_count + assigned_from_sub + assigned_sys_only + general_count
print(f"\nFinal assignments ({total}/{db.count()}):")
print(f"  From (sys,sub) pair:  {assigned_from_sub}")
print(f"  Utility:              {utility_count}")
print(f"  System only (General sub): {assigned_sys_only}")
print(f"  Full General fallback: {general_count}")

# ── Save ─────────────────────────────────────────────────────────────
no_system = sum(1 for _, _, doc in db.items() if not doc.system)
no_subsystem = sum(1 for _, _, doc in db.items() if not doc.subsystem)
print(f"\nBefore save — missing system: {no_system}, missing subsystem: {no_subsystem}")
db.save()
print(f"Saved doc_db.json with {db.count()} entities")

Resolved 618 multi-candidate entities via k-NN

Final assignments (5293/5293):
  From (sys,sub) pair:  4457
  Utility:              58
  System only (General sub): 778
  Full General fallback: 0

Before save — missing system: 0, missing subsystem: 0
Saved doc_db.json with 5293 entities


In [25]:
# ── Validation summary ───────────────────────────────────────────────
sys_dist: Counter = Counter()
sub_dist: Counter = Counter()
no_system = 0
no_subsystem = 0

for cid, sig, doc in db.items():
    if doc.system:
        sys_dist[doc.system] += 1
    else:
        no_system += 1
    if doc.subsystem:
        sub_dist[doc.subsystem] += 1
    else:
        no_subsystem += 1

print(f"Total entities: {db.count()}")
print(f"With system:    {db.count() - no_system} ({100*(db.count()-no_system)/db.count():.1f}%)")
print(f"With subsystem: {db.count() - no_subsystem} ({100*(db.count()-no_subsystem)/db.count():.1f}%)")
print(f"\n── System distribution ({'─'*40})")
for sys, cnt in sys_dist.most_common():
    print(f"  {sys:30s} {cnt:5d}")
print(f"\n── Top-25 subsystem distribution ({'─'*30})")
for sub, cnt in sub_dist.most_common(25):
    print(f"  {sub:40s} {cnt:5d}")

# ── Fallback / "General" subsystem report ────────────────────────────
general_subs = {sub: cnt for sub, cnt in sub_dist.items()
                if sub.endswith(" General") or sub == "General"}
if general_subs:
    total_general = sum(general_subs.values())
    pct = 100 * total_general / db.count()
    print(f"\n{'='*72}")
    print(f"FALLBACK SUBSYSTEMS: {total_general} entities ({pct:.1f}%) in 'General' buckets")
    print(f"{'─'*72}")
    for sub, cnt in sorted(general_subs.items(), key=lambda x: -x[1]):
        print(f"  {sub:40s} {cnt:5d}")
    print(f"{'='*72}")

    # Show a sample of entities per General bucket for quick triage
    print(f"\n  Sample entities per General bucket (up to 10 each):")
    for sub_name in sorted(general_subs, key=lambda s: -general_subs[s]):
        sample = [(doc.name, doc.kind, doc.cid)
                  for _, _, doc in db.items()
                  if doc.subsystem == sub_name]
        print(f"\n  [{sub_name}] ({len(sample)} total)")
        for ename, ekind, ecid in sorted(sample)[:10]:
            print(f"    {ekind:12s} {ename:40s} ({ecid})")
        if len(sample) > 10:
            print(f"    ... and {len(sample) - 10} more")
else:
    print("\n✓ No fallback 'General' subsystems — all entities placed.")

Total entities: 5293
With system:    5293 (100.0%)
With subsystem: 5293 (100.0%)

── System distribution (────────────────────────────────────────)
  Data Tables                     1319
  Command Interpreter              715
  World System                     423
  Object System                    401
  Utilities                        370
  Character Data Model             289
  Networking                       260
  Magic System                     185
  Social & Communication           183
  MobProg System                   157
  Affect System                    149
  Organizations & PvP              143
  Persistence                      133
  Combat System                    114
  Notes & Editor                   109
  Admin & Builder Commands          69
  Memory & Garbage Collection       56
  Quests                            55
  Character Progression             47
  Help System                       40
  Game Engine                       35
  Economy                        

In [13]:
# ── Full listing: system → subsystem → member signatures ─────────────
from collections import defaultdict as _dd

_tree: dict[str, dict[str, list[str]]] = _dd(lambda: _dd(list))
for _cid, _sig, _doc in db.items():
    _sys = _doc.system or "(none)"
    _sub = _doc.subsystem or "(none)"
    _tree[_sys][_sub].append(_sig)

for _sys in sorted(_tree):
    print(f"\n{'='*72}")
    print(f"SYSTEM: {_sys}")
    print(f"{'='*72}")
    for _sub in sorted(_tree[_sys]):
        members = sorted(_tree[_sys][_sub])
        print(f"\n  [{_sub}] ({len(members)} members)")
        for _m in members:
            print(f"    {_m}")


SYSTEM: Admin & Builder Commands

  [Act Group] (43 members)
    SIZE_HUGE
    bool CAN_USE_RSKILL(Character *ch, skill::type sn)
    bool CAN_USE_RSKILL(Character *ch, skill::type sn)
    bool HAS_EXTRACLASS(Character *ch, skill::type sn)
    bool HAS_EXTRACLASS(Character *ch, skill::type sn)
    bool completed_group(Character *ch, int gn)
    bool deduct_stamina(Character *ch, skill::type sn)
    bool deduct_stamina(Character *ch, skill::type sn)
    const skill_table_t & skill::lookup(type t)
    int can_evolve(Character *ch, skill::type type)
    int get_evolution(const Character *ch, skill::type sn)
    int get_evolution(const Character *ch, skill::type sn)
    int get_learned(const Character *ch, skill::type sn)
    int get_learned(const Character *ch, skill::type sn)
    int get_skill_cost(Character *ch, skill::type sn)
    int get_skill_cost(Character *ch, skill::type sn)
    int get_skill_level(const Character *ch, skill::type sn)
    int get_skill_level(const Character *ch, 

In [26]:
# ── Diagnostic: check pipeline state at each phase ───────────────────
print("=== Phase 1 check: file-path seed coverage ===")
print(f"  sys_cands keys: {len(sys_cands)}")
print(f"  Member hashes with cands: {sum(1 for h in hash_to_doc if h in sys_cands)}/{len(hash_to_doc)}")
print(f"  Compound IDs with cands: {sum(1 for c in compound_has_doc if c in sys_cands)}/{len(compound_has_doc)}")

# What does a typical sys_cands entry look like after broadening?
cand_sizes = [len(v) for v in sys_cands.values() if v]
if cand_sizes:
    print(f"\n=== Candidate set sizes (after broadening + utility + narrowing) ===")
    for sz in range(1, 8):
        cnt = sum(1 for v in sys_cands.values() if len(v) == sz)
        if cnt:
            print(f"  {sz} candidates: {cnt} entities")
    print(f"  >7 candidates: {sum(1 for v in sys_cands.values() if len(v) > 7)} entities")
    print(f"  avg: {np.mean(cand_sizes):.2f}, max: {max(cand_sizes)}")

# What systems are most common as candidates?
print(f"\n=== System frequency in candidate sets ===")
sys_freq = Counter()
for cands in sys_cands.values():
    for s in cands:
        sys_freq[s] += 1
for sys_name, cnt in sys_freq.most_common():
    print(f"  {sys_name:30s} appears in {cnt:5d} entities")

# Check Leiden cluster sizes
print(f"\n=== Leiden cluster sizes ===")
cl_sizes = Counter(leiden_clustering.values())
size_dist = Counter()
for cl, cnt in cl_sizes.items():
    size_dist[cnt] += 1
for sz in sorted(size_dist.keys()):
    if sz <= 10 or sz > 50:
        print(f"  size {sz:5d}: {size_dist[sz]} clusters")
print(f"  Largest cluster: {max(cl_sizes.values())} nodes")

# Check sub_cands
print(f"\n=== sub_cands pair counts ===")
pair_sizes = [len(v) for v in sub_cands.values() if v]
if pair_sizes:
    for sz in range(1, 6):
        cnt = sum(1 for v in sub_cands.values() if len(v) == sz)
        if cnt:
            print(f"  {sz} (sys,sub) pairs: {cnt} entities")
    print(f"  >5 pairs: {sum(1 for v in sub_cands.values() if len(v) > 5)} entities")
    print(f"  avg: {np.mean(pair_sizes):.2f}")

# Final assignment breakdown
print(f"\n=== Final assignments ===")
print(f"  utility: {utility_count}")
print(f"  from (sys,sub): {assigned_from_sub}")
print(f"  sys only: {assigned_sys_only}")
print(f"  general fallback: {general_count}")

# Check: how many entities ended up in their file-path seeded system?
match = 0
mismatch = 0
no_seed = 0
for cid, sig, doc in db.items():
    _, mhash = dp.EntityID.split(doc.mid)
    key = mhash if mhash else doc.cid
    files = CID_TO_FILES.get(doc.cid, set())
    file_systems = set()
    for fn in files:
        for s in FILE_TO_SYSTEMS.get(fn, []):
            file_systems.add(s)
    if not file_systems:
        no_seed += 1
    elif doc.system in file_systems:
        match += 1
    else:
        mismatch += 1
print(f"\n=== File-path agreement ===")
print(f"  Matches file-path: {match}")
print(f"  Disagrees:         {mismatch}")
print(f"  No file seed:      {no_seed}")

=== Phase 1 check: file-path seed coverage ===
  sys_cands keys: 5260
  Member hashes with cands: 4964/5015
  Compound IDs with cands: 216/216

=== Candidate set sizes (after broadening + utility + narrowing) ===
  1 candidates: 5260 entities
  >7 candidates: 0 entities
  avg: 1.00, max: 1

=== System frequency in candidate sets ===
  Data Tables                    appears in  1329 entities
  Command Interpreter            appears in   768 entities
  Object System                  appears in   383 entities
  World System                   appears in   379 entities
  Utilities                      appears in   321 entities
  Character Data Model           appears in   299 entities
  Networking                     appears in   253 entities
  Magic System                   appears in   230 entities
  MobProg System                 appears in   163 entities
  Affect System                  appears in   149 entities
  Social & Communication         appears in   148 entities
  Organizations 